# 03 — LoRA From Scratch (no `peft`)

Part of [bert-lora-finetuning](../README.md). Not part of the BERT comparison in 01/02 — this notebook
demonstrates the actual mechanism `peft` implements, using [`src/lora_from_scratch.py`](../src/lora_from_scratch.py)
directly. No GPU needed, runs anywhere.

This exists so there's code you wrote yourself to point to, not just a config object, when the question
is "how does LoRA actually work."


In [ ]:
import sys
sys.path.insert(0, "..")

import torch
import torch.nn as nn
from src.lora_from_scratch import LoRALinear


## 1. At init, LoRA changes nothing (B starts at zero)

In [ ]:
torch.manual_seed(0)
frozen_layer = nn.Linear(768, 768)   # same hidden size as one BERT attention projection
lora_layer = LoRALinear(frozen_layer, r=8, alpha=16)

x = torch.randn(2, 5, 768)  # (batch, seq_len, hidden)

with torch.no_grad():
    out_wrapped = lora_layer(x)
    out_frozen_only = frozen_layer(x)

print("Max difference at init (should be exactly 0.0):",
      (out_wrapped - out_frozen_only).abs().max().item())

print(f"Trainable (A + B): {lora_layer.trainable_parameter_count():,}")
print(f"Frozen (W0):       {lora_layer.frozen_parameter_count():,}")


## 2. Gradients only flow into A and B

In [ ]:
lora_layer.zero_grad()
out = lora_layer(x)
out.sum().backward()

print("A.grad is None?", lora_layer.A.grad is None)
print("B.grad is None?", lora_layer.B.grad is None)
print("base.weight.grad is None (should be True — frozen)?", lora_layer.base.weight.grad is None)


## 3. Merging: the deployment story

In [ ]:
# Take one manual gradient step so B moves away from zero, then merge B@A back into a plain nn.Linear.
with torch.no_grad():
    lora_layer.A -= 0.1 * lora_layer.A.grad
    lora_layer.B -= 0.1 * lora_layer.B.grad

merged_layer = lora_layer.merge()

with torch.no_grad():
    out_unmerged = lora_layer(x)
    out_merged = merged_layer(x)

print("Max difference, unmerged vs merged (should be tiny — float32 rounding, not a bug):",
      (out_unmerged - out_merged).abs().max().item())
print()
print("merged_layer is a plain nn.Linear — same forward-pass cost as the original frozen layer.")
print("This is why LoRA adds zero inference latency once merged: A and B never touch the model at")
print("inference time, only W0 + the folded-in update do.")


## What this maps to in the real model

`peft`'s `get_peft_model()` does this same wrap — freeze, add `A`/`B`, scale by `alpha/r` — for every
`target_modules` match (here, `query` and `value`) across all 12 BERT layers, and its `merge_and_unload()`
does the same fold-back-in shown above. The tests in [`tests/test_lora_from_scratch.py`](../tests/test_lora_from_scratch.py)
check all three properties demonstrated here: identical output at init, gradient isolation, and
merge-equivalence. They run in CI on every push.
